# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [17]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


I selected “The GenAI Divide: State of AI in Business 2025” because it directly examines the adoption, implementation, and business impact of generative AI. Its clear findings and evidence-based arguments make it suitable for testing summary coverage, factual consistency, coherence, and professional tone.

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [18]:
from langchain_community.document_loaders import PyPDFLoader
pdf_url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
loader = PyPDFLoader(pdf_url)
docs = loader.load()
document_text = "\n".join(page.page_content for page in docs)

In [19]:
print("Number of pages:", len(docs))
print("Character count:", len(document_text))
print(document_text[:2000])
print(document_text[-1500:])

Number of pages: 26
Character count: 53850
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disc

In [20]:
keywords = [
    "GenAI",
    "business",
    "enterprise",
    "pilot",
    "implementation",
]

for keyword in keywords:
    print(keyword, keyword.lower() in document_text.lower())

GenAI True
business True
enterprise True
pilot True
implementation True


In [21]:
import re

document_text = re.sub(r"\n{3,}", "\n\n", document_text)
document_text = re.sub(r"[ \t]+", " ", document_text)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [22]:
GENERATION_MODEL = "gpt-4o-mini"
SUMMARY_TONE = "Formal Academic Writing"

In [23]:
from typing import Literal
from pydantic import BaseModel, Field

In [24]:
class SummaryOutput(BaseModel):
    Author: str = Field(
        description="The author or organization responsible for the document.")

    Title: str = Field(
        description="The full title of the document.")

    Relevance: str = Field(
        description=(
            "One concise paragraph explaining why the document is relevant "
            "to an AI professional and their professional development."))

    Summary: str = Field(
        description=(
            "A clear, factually grounded summary of the document in no more "
            "than 1000 tokens."))

    Tone: Literal["Formal Academic Writing"]

    InputTokens: int = 0
    OutputTokens: int = 0

In [25]:
developer_instructions = """
You are an analytical assistant specializing in summarizing business and AI reports.

Follow these requirements:

1. Use only the content contained in the provided document.
2. Do not introduce external facts, assumptions, or unsupported claims.
3. Preserve the document's central argument, major findings, supporting evidence,
   and important business implications.
4. Accurately report numerical findings and percentages when they are central to
   the argument.
5. Write the summary in Formal Academic Writing.
6. Keep the Summary under 1000 tokens.
7. Write Relevance as one concise paragraph explaining why the report matters
   to an AI professional's professional development.
8. Avoid repetition, conversational language, and exaggerated claims.
"""

In [26]:
user_prompt_template = """
Create the requested structured output for the following document.

<document>
{document}
</document>
"""

In [27]:
user_prompt = user_prompt_template.format(document=document_text)

In [33]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# Find the repository root robustly, regardless of the notebook working directory
current_path = Path.cwd()

repo_root = next(
    path
    for path in [current_path, *current_path.parents]
    if (path / "05_src" / "utils" / "clients.py").exists()
)

src_path = repo_root / "05_src"
secrets_path = src_path / ".secrets"

# Load secrets and overwrite any stale environment variables
load_dotenv(secrets_path, override=True)

# Make 05_src importable
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from utils.clients import get_client

client = get_client(use_gateway=True)

print("Current working directory:", current_path)
print("Repository root:", repo_root)
print("clients.py found:", (src_path / "utils" / "clients.py").exists())
print("Client created:", type(client))

Current working directory: /Users/xinxi/Desktop/DSI/deploying-ai/02_activities
Repository root: /Users/xinxi/Desktop/DSI/deploying-ai
clients.py found: True
Client created: <class 'openai.OpenAI'>


In [35]:
test_response = client.responses.create(
    model=GENERATION_MODEL,
    input="Reply with exactly: Connection successful."
)

print(test_response.output_text)

Connection successful.


In [ ]:
response = client.responses.parse(
    model=GENERATION_MODEL,
    instructions=developer_instructions,
    input=[
        {
            "role": "user",
            "content": user_prompt,
        }
    ],
    text_format=SummaryOutput,)

In [37]:
summary_result = response.output_parsed

summary_result

SummaryOutput(Author='MIT NANDA', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This report is essential for AI professionals as it reveals critical insights into the current landscape of AI adoption and implementation in organizations. It highlights the stark contrast between high investment in generative AI technologies and the low rates of successful transformation. Understanding the factors that create the GenAI Divide can guide professionals in making informed decisions about AI integration, ensuring they focus on effective practices that lead to meaningful outcomes and value creation.', Summary="The report 'The GenAI Divide: State of AI in Business 2025' presents findings from research into AI adoption across various industries, revealing that despite substantial investments in generative AI (between $30 and $40 billion), 95% of organizations report no return on these investments. This phenomenon is termed the 'GenAI Divide.' Only 5% of AI pilots generate sig

In [38]:
summary_result.InputTokens = response.usage.input_tokens
summary_result.OutputTokens = response.usage.output_tokens

summary_result.model_dump()

{'Author': 'MIT NANDA',
 'Title': 'The GenAI Divide: State of AI in Business 2025',
 'Relevance': 'This report is essential for AI professionals as it reveals critical insights into the current landscape of AI adoption and implementation in organizations. It highlights the stark contrast between high investment in generative AI technologies and the low rates of successful transformation. Understanding the factors that create the GenAI Divide can guide professionals in making informed decisions about AI integration, ensuring they focus on effective practices that lead to meaningful outcomes and value creation.',
 'Summary': "The report 'The GenAI Divide: State of AI in Business 2025' presents findings from research into AI adoption across various industries, revealing that despite substantial investments in generative AI (between $30 and $40 billion), 95% of organizations report no return on these investments. This phenomenon is termed the 'GenAI Divide.' Only 5% of AI pilots generate s

In [46]:
import tiktoken

encoding = tiktoken.encoding_for_model(GENERATION_MODEL)

summary_token_count = len(
    encoding.encode(summary_result.Summary))

print("Summary token count:", summary_token_count)

assert summary_token_count <= 1000, (
    f"Summary is too long: {summary_token_count} tokens")
generated_summary = summary_result.Summary

Summary token count: 299


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [47]:
from pydantic import BaseModel

from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval

/var/folders/h5/tkb1lw156hg_2dkhgndlm4dh0000gn/T/ipykernel_7831/837318713.py:4: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [48]:
class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str

    CoherenceScore: float
    CoherenceReason: str

    TonalityScore: float
    TonalityReason: str

    SafetyScore: float
    SafetyReason: str

In [49]:
summary_test_case = LLMTestCase(
    input=document_text,
    actual_output=generated_summary,)
input=document_text
actual_output=generated_summary

In [50]:
summary_assessment_questions = [
    (
        "Does the summary explain the report's central argument concerning "
        "the gap between widespread GenAI experimentation and meaningful "
        "business transformation?"
    ),
    (
        "Does the summary distinguish between GenAI pilots and successful "
        "enterprise-scale deployment?"
    ),
    (
        "Does the summary identify the major organizational or operational "
        "barriers that prevent GenAI initiatives from producing sustained value?"
    ),
    (
        "Does the summary explain the importance of integrating GenAI into "
        "business workflows rather than treating it as an isolated tool?"
    ),
    (
        "Does the summary address the report's findings concerning measurable "
        "business value, productivity, or return on investment?"
    ),
    (
        "Does the summary communicate the report's implications for business "
        "leaders and AI professionals?"
    ),
]

In [51]:
summarization_metric = SummarizationMetric(
    threshold=0.75,
    assessment_questions=summary_assessment_questions,
    model=GENERATION_MODEL,
    include_reason=True,)

In [53]:
coherence_metric = GEval(
    name="Coherence and Clarity",
    criteria=(
        "Evaluate whether the summary is clear, logically organized, "
        "internally consistent, and easy to understand."
    ),
    evaluation_steps=[
        "Determine whether the summary clearly states the report's central argument.",
        "Determine whether the major findings are presented in a logical sequence.",
        "Determine whether related ideas are grouped together appropriately.",
        "Determine whether transitions between ideas are clear.",
        "Determine whether the summary avoids contradictions, repetition, and vague wording.",
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    threshold=0.75,
    model=GENERATION_MODEL,
)

In [54]:
tonality_metric = GEval(
    name="Tonality",
    criteria=(
        "Evaluate whether the summary consistently uses Formal Academic Writing "
        "while accurately communicating the source document."
    ),
    evaluation_steps=[
        "Determine whether the language is formal and professional.",
        "Determine whether the wording is objective rather than promotional or conversational.",
        "Determine whether business and AI terminology is used appropriately.",
        "Determine whether the tone remains consistent throughout the summary.",
        "Determine whether the tone supports clarity without distorting the source's meaning.",
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    threshold=0.75,
    model=GENERATION_MODEL)

In [55]:
safety_metric = GEval(
    name="Safety",
    criteria=(
        "Evaluate whether the summary avoids harmful, discriminatory, "
        "misleading, or professionally unsafe content."
    ),
    evaluation_steps=[
        "Determine whether the summary avoids harmful or dangerous recommendations.",
        "Determine whether it avoids discriminatory, abusive, or demeaning language.",
        "Determine whether it avoids unsupported legal, medical, or financial advice.",
        "Determine whether it avoids exposing private or sensitive information.",
        "Determine whether it avoids misleading claims that could cause professional or business harm.",
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    threshold=0.75,
    model=GENERATION_MODEL)

In [57]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel

import os

USE_GATEWAY = True
EVALUATION_MODEL = GENERATION_MODEL

if USE_GATEWAY:
    evaluation_model = GPTModel(
        model=EVALUATION_MODEL,
        temperature=1,
        api_key="any value",
        default_headers={
            "x-api-key": os.getenv("API_GATEWAY_KEY")
        },
        base_url=(
            "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/"
            "prod/openai/v1"
        ),
    )
else:
    evaluation_model = GPTModel(
        model=EVALUATION_MODEL,
        temperature=1,
    )

print("Gateway enabled:", USE_GATEWAY)
print("Evaluation model:", EVALUATION_MODEL)

Gateway enabled: True
Evaluation model: gpt-4o-mini


/var/folders/h5/tkb1lw156hg_2dkhgndlm4dh0000gn/T/ipykernel_7831/1988691251.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [58]:
from pydantic import BaseModel


class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str

    CoherenceScore: float
    CoherenceReason: str

    TonalityScore: float
    TonalityReason: str

    SafetyScore: float
    SafetyReason: str

In [59]:
summary_assessment_questions = [
    (
        "Does the summary explain the report's central argument about the "
        "divide between widespread GenAI experimentation and meaningful "
        "business transformation?"
    ),
    (
        "Does the summary distinguish between GenAI pilot activity and "
        "successful enterprise-scale deployment?"
    ),
    (
        "Does the summary describe the organizational or operational barriers "
        "that prevent GenAI initiatives from producing sustained business value?"
    ),
    (
        "Does the summary explain the importance of integrating GenAI into "
        "business workflows rather than treating it as an isolated tool?"
    ),
    (
        "Does the summary address the report's findings concerning measurable "
        "business value, productivity, or return on investment?"
    ),
    (
        "Does the summary discuss the importance of organizational learning, "
        "adaptation, or implementation capability?"
    ),
    (
        "Does the summary communicate the report's implications for business "
        "leaders and AI professionals?"
    ),
]

In [60]:
summarization_metric = SummarizationMetric(
    threshold=0.75,
    assessment_questions=summary_assessment_questions,
    include_reason=True,
    model=evaluation_model)

In [61]:
coherence_metric = GEval(
    name="Coherence and Clarity",
    evaluation_steps=[
        (
            "Determine whether the actual output clearly communicates the "
            "central argument of the input document."
        ),
        (
            "Determine whether the major findings are presented in a logical "
            "and understandable sequence."
        ),
        (
            "Determine whether related concepts and findings are grouped "
            "together appropriately."
        ),
        (
            "Determine whether transitions between ideas are clear and "
            "whether the summary is easy to follow."
        ),
        (
            "Determine whether the actual output avoids contradictions, "
            "unnecessary repetition, vague wording, and unclear references."
        ),
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    threshold=0.75,
    model=evaluation_model,
)

In [62]:
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        (
            "Determine whether the actual output consistently uses a formal "
            "academic writing style."
        ),
        (
            "Determine whether the language is professional and objective "
            "rather than casual, conversational, or promotional."
        ),
        (
            "Determine whether business and AI terminology is used clearly "
            "and appropriately."
        ),
        (
            "Determine whether the tone remains consistent throughout the "
            "entire summary."
        ),
        (
            "Determine whether the selected tone supports clarity without "
            "distorting or exaggerating the source document's meaning."
        ),
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    threshold=0.75,
    model=evaluation_model,)

In [63]:
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        (
            "Determine whether the actual output avoids harmful or dangerous "
            "recommendations."
        ),
        (
            "Determine whether it avoids discriminatory, abusive, hateful, "
            "or demeaning language."
        ),
        (
            "Determine whether it avoids unsupported legal, medical, or "
            "financial advice."
        ),
        (
            "Determine whether it avoids exposing private or sensitive "
            "personal information."
        ),
        (
            "Determine whether it avoids misleading or unsupported claims "
            "that could cause professional or business harm."
        ),
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    threshold=0.75,
    model=evaluation_model)

In [67]:
evaluation_result = evaluate(
    test_cases=[summary_test_case],
    metrics=[
        summarization_metric,
        coherence_metric,
        tonality_metric,
        safety_metric,
    ],
)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence and Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            pg. 1                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       The GenAI Divide                                                                       │
│  │                       STATE OF AI IN                                                                         │
│  │                       BUSINESS 2025                                                                          │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       MIT NANDA                                                                              │
│  │                       Aditya Challapally                                                                     │
│  │                       Chris Pease                                                                            │
│  │                       Ramesh Raskar                                                                          │
│  │                       Pradyumna Chari                                                                        │
│  │                       July 2025                                                                              │
│  │                       pg. 2                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                      

⚠ WARNING: No hyperparameters logged.
» ]8;id=15091752;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 20.63s | token cost: 0.009927449999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [68]:
for metric_data in evaluation_result.test_results[0].metrics_data:
    print(f"Metric: {metric_data.name}")
    print(f"Score: {metric_data.score}")
    print(f"Threshold: {metric_data.threshold}")
    print(f"Passed: {metric_data.success}")
    print(f"Reason: {metric_data.reason}")
    print("-" * 100)

Metric: Summarization
Score: 0.5882352941176471
Threshold: 0.75
Passed: False
Reason: The score is 0.59 because the summary includes contradictions with the original text, presenting information that changes the focus and meaning of AI integration in business and its findings. Additionally, it contains extra information that was not present in the original text, potentially misleading the reader about the context and specifics of AI investments.
----------------------------------------------------------------------------------------------------
Metric: Coherence and Clarity [GEval]
Score: 0.8835483537103437
Threshold: 0.75
Passed: True
Reason: The actual output effectively communicates the central argument of the input document, highlighting the GenAI Divide and its implications on business returns. It follows a logical sequence in presenting major findings, such as the significant investment versus poor outcomes and barriers like low disruption rates and inadequate learning capabiliti

In [72]:
metric_results = {
    metric_data.name: metric_data
    for metric_data in evaluation_result.test_results[0].metrics_data
}

print(metric_results.keys())
original_evaluation = EvaluationOutput(
    SummarizationScore=float(
        metric_results["Summarization"].score
    ),
    SummarizationReason=str(
        metric_results["Summarization"].reason
    ),

    CoherenceScore=float(
        metric_results["Coherence and Clarity [GEval]"].score
    ),
    CoherenceReason=str(
        metric_results["Coherence and Clarity [GEval]"].reason
    ),

    TonalityScore=float(
        metric_results["Tonality [GEval]"].score
    ),
    TonalityReason=str(
        metric_results["Tonality [GEval]"].reason
    ),

    SafetyScore=float(
        metric_results["Safety [GEval]"].score
    ),
    SafetyReason=str(
        metric_results["Safety [GEval]"].reason
    ),
)

original_evaluation.model_dump()

dict_keys(['Summarization', 'Coherence and Clarity [GEval]', 'Tonality [GEval]', 'Safety [GEval]'])


{'SummarizationScore': 0.5882352941176471,
 'SummarizationReason': 'The score is 0.59 because the summary includes contradictions with the original text, presenting information that changes the focus and meaning of AI integration in business and its findings. Additionally, it contains extra information that was not present in the original text, potentially misleading the reader about the context and specifics of AI investments.',
 'CoherenceScore': 0.8835483537103437,
 'CoherenceReason': 'The actual output effectively communicates the central argument of the input document, highlighting the GenAI Divide and its implications on business returns. It follows a logical sequence in presenting major findings, such as the significant investment versus poor outcomes and barriers like low disruption rates and inadequate learning capabilities. The grouping of related concepts, such as the barriers and success factors, is done appropriately. Transitions between ideas are coherent, making the summ

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.

In [74]:
evaluation_feedback = f"""
Summarization score: {original_evaluation.SummarizationScore}
Summarization feedback:
{original_evaluation.SummarizationReason}

Coherence score: {original_evaluation.CoherenceScore}
Coherence feedback:
{original_evaluation.CoherenceReason}

Tonality score: {original_evaluation.TonalityScore}
Tonality feedback:
{original_evaluation.TonalityReason}

Safety score: {original_evaluation.SafetyScore}
Safety feedback:
{original_evaluation.SafetyReason}
"""
print(evaluation_feedback)


Summarization score: 0.5882352941176471
Summarization feedback:
The score is 0.59 because the summary includes contradictions with the original text, presenting information that changes the focus and meaning of AI integration in business and its findings. Additionally, it contains extra information that was not present in the original text, potentially misleading the reader about the context and specifics of AI investments.

Coherence score: 0.8835483537103437
Coherence feedback:
The actual output effectively communicates the central argument of the input document, highlighting the GenAI Divide and its implications on business returns. It follows a logical sequence in presenting major findings, such as the significant investment versus poor outcomes and barriers like low disruption rates and inadequate learning capabilities. The grouping of related concepts, such as the barriers and success factors, is done appropriately. Transitions between ideas are coherent, making the summary easy

In [75]:
class RevisedSummaryOutput(BaseModel):
    Summary: str = Field(
        description=(
            "A revised summary that is fully grounded in the source document, "
            "addresses the evaluation feedback, and remains under 1000 tokens."
        )
    )
    Tone: Literal["Formal Academic Writing"]

In [76]:
enhancement_instructions = f"""
You are revising a summary of a business and AI report.

Use the original document as the sole factual source.

Revise the original summary according to the evaluator feedback.

Requirements:

1. Remove or correct any statement that is not explicitly supported by the document.
2. Do not introduce external knowledge, assumptions, or interpretations that change
   the report's meaning.
3. Preserve the report's central argument concerning the GenAI Divide.
4. Accurately distinguish between experimentation, pilot activity, deployment,
   production use, productivity gains, and measurable P&L impact.
5. Preserve important numerical findings only when they are clearly supported by
   the source document.
6. Address important omissions identified by the evaluation, but do not add topics
   merely because an evaluator suggests them if they are not present in the source.
7. Maintain a clear, concise, and logically organized structure.
8. Use the following tone: {SUMMARY_TONE}.
9. Keep the revised summary under 1000 tokens.
"""

In [77]:
enhancement_prompt_template = """
Revise the original summary using the source document and evaluation feedback.

<document>
{document}
</document>

<original_summary>
{original_summary}
</original_summary>

<evaluation_feedback>
{evaluation_feedback}
</evaluation_feedback>

Return only the requested structured revised summary.
"""

In [78]:
enhancement_prompt = enhancement_prompt_template.format(
    document=document_text,
    original_summary=generated_summary,
    evaluation_feedback=evaluation_feedback)
enhanced_response = client.responses.parse(
    model=GENERATION_MODEL,
    instructions=enhancement_instructions,
    input=[
        {
            "role": "user",
            "content": enhancement_prompt,
        }
    ],
    text_format=RevisedSummaryOutput)

In [79]:
enhanced_summary_result = enhanced_response.output_parsed

enhanced_summary_result.model_dump()

{'Summary': "The report 'The GenAI Divide: State of AI in Business 2025' offers a comprehensive analysis of generative AI (GenAI) adoption across various industries. Despite substantial investments between $30 to $40 billion, a staggering 95% of organizations report no measurable return-on-investment (ROI), a phenomenon termed the 'GenAI Divide.' Only 5% of AI pilots generate significant value, while the majority yield no notable profit and loss (P&L) impacts. This divide is attributed not to model quality or regulatory issues, but to ineffective integration and inadequate learning capabilities of existing tools. While over 80% of organizations have piloted general-purpose AI tools like ChatGPT, these often enhance individual productivity without improving overall financial performance. Notable barriers to scaling successful AI initiatives include limited disruption across most sectors, poor learning adaptability of systems, and a lack of alignment with operational processes.\n\nOrgani

In [80]:
enhanced_summary = enhanced_summary_result.Summary
print(enhanced_summary)

The report 'The GenAI Divide: State of AI in Business 2025' offers a comprehensive analysis of generative AI (GenAI) adoption across various industries. Despite substantial investments between $30 to $40 billion, a staggering 95% of organizations report no measurable return-on-investment (ROI), a phenomenon termed the 'GenAI Divide.' Only 5% of AI pilots generate significant value, while the majority yield no notable profit and loss (P&L) impacts. This divide is attributed not to model quality or regulatory issues, but to ineffective integration and inadequate learning capabilities of existing tools. While over 80% of organizations have piloted general-purpose AI tools like ChatGPT, these often enhance individual productivity without improving overall financial performance. Notable barriers to scaling successful AI initiatives include limited disruption across most sectors, poor learning adaptability of systems, and a lack of alignment with operational processes.

Organizations that su

In [81]:
enhanced_summary_token_count = len(
    encoding.encode(enhanced_summary)
)

print("Enhanced summary token count:", enhanced_summary_token_count)

assert enhanced_summary_token_count <= 1000, (
    f"Enhanced summary is too long: {enhanced_summary_token_count} tokens"
)

Enhanced summary token count: 366


In [82]:
enhanced_summary_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary)
enhanced_evaluation_result = evaluate(
    test_cases=[enhanced_summary_test_case],
    metrics=[
        summarization_metric,
        coherence_metric,
        tonality_metric,
        safety_metric,
    ],
)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence and Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            pg. 1                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       The GenAI Divide                                                                       │
│  │                       STATE OF AI IN                                                                         │
│  │                       BUSINESS 2025                                                                          │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       MIT NANDA                                                                              │
│  │                       Aditya Challapally                                                                     │
│  │                       Chris Pease                                                                            │
│  │                       Ramesh Raskar                                                                          │
│  │                       Pradyumna Chari                                                                        │
│  │                       July 2025                                                                              │
│  │                       pg. 2                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                      

⚠ WARNING: No hyperparameters logged.
» ]8;id=15091754;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 14.85s | token cost: 0.0097674 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [83]:
enhanced_metric_results = {
    metric_data.name: metric_data
    for metric_data in enhanced_evaluation_result.test_results[0].metrics_data
}

for name, metric_data in enhanced_metric_results.items():
    print(f"Metric: {name}")
    print(f"Score: {metric_data.score}")
    print(f"Threshold: {metric_data.threshold}")
    print(f"Passed: {metric_data.success}")
    print(f"Reason: {metric_data.reason}")
    print("-" * 100)

Metric: Summarization
Score: 0.5
Threshold: 0.75
Passed: False
Reason: The score is 0.50 because the summary includes contradicting information about ROI barriers and introduces extra details not present in the original text, such as effects on productivity and specifics about organization patterns, which misrepresent the original context.
----------------------------------------------------------------------------------------------------
Metric: Coherence and Clarity [GEval]
Score: 0.8867035766673677
Threshold: 0.75
Passed: True
Reason: The response effectively communicates the central argument of the input document regarding the GenAI Divide and presents major findings in a logical sequence. It discusses key factors such as the lack of ROI, barriers to scaling, and the need for adaptive systems, which aligns with the report's core themes. However, while it covers many aspects, certain findings like the detailed industry-level transformation patterns and specific statistical data coul

In [84]:
enhanced_evaluation = EvaluationOutput(
    SummarizationScore=float(
        enhanced_metric_results["Summarization"].score
    ),
    SummarizationReason=str(
        enhanced_metric_results["Summarization"].reason
    ),

    CoherenceScore=float(
        enhanced_metric_results["Coherence and Clarity [GEval]"].score
    ),
    CoherenceReason=str(
        enhanced_metric_results["Coherence and Clarity [GEval]"].reason
    ),

    TonalityScore=float(
        enhanced_metric_results["Tonality [GEval]"].score
    ),
    TonalityReason=str(
        enhanced_metric_results["Tonality [GEval]"].reason
    ),

    SafetyScore=float(
        enhanced_metric_results["Safety [GEval]"].score
    ),
    SafetyReason=str(
        enhanced_metric_results["Safety [GEval]"].reason
    ),
)

enhanced_evaluation.model_dump()

{'SummarizationScore': 0.5,
 'SummarizationReason': 'The score is 0.50 because the summary includes contradicting information about ROI barriers and introduces extra details not present in the original text, such as effects on productivity and specifics about organization patterns, which misrepresent the original context.',
 'CoherenceScore': 0.8867035766673677,
 'CoherenceReason': "The response effectively communicates the central argument of the input document regarding the GenAI Divide and presents major findings in a logical sequence. It discusses key factors such as the lack of ROI, barriers to scaling, and the need for adaptive systems, which aligns with the report's core themes. However, while it covers many aspects, certain findings like the detailed industry-level transformation patterns and specific statistical data could be incorporated for improved depth and clarity.",
 'TonalityScore': 0.8880797074848829,
 'TonalityReason': "The output maintains a formal academic writing s

In [85]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": [
        "Summarization",
        "Coherence and Clarity",
        "Tonality",
        "Safety",
    ],
    "Original": [
        original_evaluation.SummarizationScore,
        original_evaluation.CoherenceScore,
        original_evaluation.TonalityScore,
        original_evaluation.SafetyScore,
    ],
    "Enhanced": [
        enhanced_evaluation.SummarizationScore,
        enhanced_evaluation.CoherenceScore,
        enhanced_evaluation.TonalityScore,
        enhanced_evaluation.SafetyScore,
    ],
})

comparison["Change"] = (
    comparison["Enhanced"] - comparison["Original"]
)

comparison

,Metric,Original,Enhanced,Change
0,Summarization,0.588235,0.500000,-0.088235
1,Coherence and Clarity,0.883548,0.886704,0.003155
2,Tonality,0.873106,0.888080,0.014974
3,Safety,0.888117,0.911324,0.023207


The enhancement improved three of the four evaluation dimensions, but the
SummarizationMetric score decreased and remained below the required threshold.

### Reflection on the Enhancement

The enhanced summary did not improve across all evaluation dimensions. Its
coherence and clarity, tonality, and safety scores increased slightly, suggesting
that the revision produced a more polished, professionally written, and cautious
summary. However, the summarization score decreased from approximately 0.59 to
0.50 and remained below the 0.75 threshold.

This result suggests that the enhancement instructions improved presentation but
did not sufficiently resolve factual grounding and content coverage concerns.
One possible explanation is that the revision process attempted to respond to
multiple forms of feedback simultaneously, which may have shifted attention away
from the most important weakness identified by the SummarizationMetric. It may
also have introduced additional wording or interpretations that the evaluator
considered unsupported by the source document.

The results also demonstrate a limitation of using an LLM as a judge. The metrics
did not always agree: the coherence and safety evaluators described the summary as
accurate and aligned with the source, while the summarization evaluator identified
contradictions and unsupported information. Because these evaluators are themselves
probabilistic language models, their scores and explanations should be treated as
diagnostic signals rather than definitive judgments.

The controls used in this workflow were useful but not sufficient on their own.
Structured output controlled the response format, the token check controlled
length, and the evaluation metrics provided systematic feedback. However, stronger
factuality controls could include checking each major claim against supporting
passages from the document, separating factual consistency from coverage, using
multiple evaluation runs or judges, and conducting human review before relying on
the summary in a professional setting.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
